## tag-hierarchy.ipynb

Builds a parent-tag mapping from the MusicBrainz flat tag list.

MusicBrainz has no formal tag taxonomy, so we construct one in two steps:
1. **Substring heuristic** — if tag A contains tag B as a whole-word substring, B is a candidate parent of A (e.g. `death metal` → `metal`)
2. **Manual audit** — the generated `tag_parents.csv` is reviewed and cleaned before use in `weights-tag-parents.ipynb`

### Outputs
| File | Description |
|---|---|
| `data/tag_parents.csv` | Child → parent mapping for audit/editing |
| `data/tag_parent_counts.csv` | How many albums each parent covers (for prioritising audit) |

In [1]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict

DATA_DIR = '../data'

### 1. Load tags and album tag counts

In [2]:
# All 237k tags with their string names
tags = pd.read_parquet(f'{DATA_DIR}/mb_tag.parquet')
tags['name'] = tags['name'].str.strip().str.lower()
print(f'Total tags: {len(tags):,}')

# Album tag counts — tells us which tags actually appear in the model
album_tags = pd.read_parquet(f'{DATA_DIR}/mb_album_tag.parquet')
tag_album_counts = album_tags.groupby('tag_id')['album_id'].nunique().rename('n_albums')

# Merge counts onto tag names
tags = tags.merge(tag_album_counts.reset_index(), left_on='id', right_on='tag_id', how='left')
tags['n_albums'] = tags['n_albums'].fillna(0).astype(int)

# Only tags that survive the model's pruning threshold (>=10 albums)
active_tags = tags[tags['n_albums'] >= 10].copy()
print(f'Active tags (>=10 albums): {len(active_tags):,}')
active_tags.sort_values('n_albums', ascending=False).head(20)

Total tags: 237,718
Active tags (>=10 albums): 3,041


,id,name,tag_id,n_albums
5,7,rock,7.0,326759
7,11,electronic,11.0,255137
15,19,pop,19.0,156389
61,71,jazz,71.0,108245
145,166,experimental,166.0,83320
48,58,ambient,58.0,68781
11,15,classical,15.0,63649
199,235,hip hop,235.0,54395
796,1091,pop rock,1091.0,45895
78,88,punk,88.0,41533


### 2. Build substring parent map

For each active tag, find all shorter active tags whose name appears as a **whole word** within it.  
Among candidates, pick the **most specific** (longest) parent — so `death metal` maps to `metal`, not `rock`.

In [3]:
# Build set of active tag names sorted by length descending (most specific first)
active_names = active_tags.set_index('name')['n_albums'].to_dict()
sorted_names = sorted(active_names.keys(), key=len, reverse=True)

def find_parent(tag_name, all_names):
    """Return the longest active tag that is a whole-word substring of tag_name,
    excluding the tag itself."""
    for candidate in all_names:
        if candidate == tag_name:
            continue
        if len(candidate) >= len(tag_name):
            continue
        # Whole-word match using word boundaries
        pattern = r'\b' + re.escape(candidate) + r'\b'
        if re.search(pattern, tag_name):
            return candidate
    return None

print('Building parent map (this takes ~1–2 min for the full active tag set)...')

rows = []
for tag_name in sorted_names:
    parent = find_parent(tag_name, sorted_names)
    if parent is not None:
        rows.append({
            'child':         tag_name,
            'parent':        parent,
            'child_albums':  active_names[tag_name],
            'parent_albums': active_names[parent],
        })

parent_map = pd.DataFrame(rows).sort_values('child_albums', ascending=False)
print(f'Child → parent pairs found: {len(parent_map):,}')
parent_map.head(30)

Building parent map (this takes ~1–2 min for the full active tag set)...
Child → parent pairs found: 1,495


,child,parent,child_albums,parent_albums
1388,pop rock,rock,45895,326759
356,alternative rock,alternative,38401,2528
1117,indie rock,indie,37541,5809
1273,synth-pop,synth,29497,85
1269,hard rock,rock,23626,326759
934,heavy metal,metal,21339,27147
937,black metal,metal,19688,27147
1271,folk rock,rock,19453,326759
363,psychedelic rock,psychedelic,19379,9659
820,dark ambient,ambient,15691,68781


### 3. Inspect false positives

Some substring matches are musically wrong — `new wave` → `wave`, `indie pop` → `pop` (debatable), `post-rock` → `rock` (correct).  
Review the top cases before saving.

In [4]:
# Pairs where child has many albums — highest priority to audit
print('=== Top 50 by child album count (audit these first) ===')
parent_map.head(50)[['child', 'parent', 'child_albums', 'parent_albums']]

=== Top 50 by child album count (audit these first) ===


,child,parent,child_albums,parent_albums
1388,pop rock,rock,45895,326759
356,alternative rock,alternative,38401,2528
1117,indie rock,indie,37541,5809
1273,synth-pop,synth,29497,85
1269,hard rock,rock,23626,326759
934,heavy metal,metal,21339,27147
937,black metal,metal,19688,27147
1271,folk rock,rock,19453,326759
363,psychedelic rock,psychedelic,19379,9659
820,dark ambient,ambient,15691,68781


In [5]:
# How many children each parent collects
parent_summary = (
    parent_map.groupby('parent')
    .agg(n_children=('child', 'count'), total_child_albums=('child_albums', 'sum'))
    .sort_values('total_child_albums', ascending=False)
)
print('=== Parents ranked by total child album coverage ===')
parent_summary.head(30)

=== Parents ranked by total child album coverage ===


,n_children,total_child_albums
parent,,
rock,63,163875
metal,42,80438
indie,6,51553
alternative,24,45744
synth,7,32243
progressive,17,24237
house,19,24151
jazz,27,22358
contemporary,14,21712


### 4. Save for manual audit

Edit `tag_parents.csv` directly:
- **Delete rows** where the mapping is wrong (false positives)
- **Edit the `parent` column** to reassign a child to a different parent
- **Add rows** manually for mappings the substring heuristic missed

The `tag_parent_counts.csv` is reference-only — do not edit it.

In [6]:
# Save parent map for audit
out_path = f'{DATA_DIR}/tag_parents.csv'
parent_map[['child', 'parent', 'child_albums', 'parent_albums']].to_csv(out_path, index=False)
print(f'Saved {len(parent_map):,} pairs → {out_path}')

# Save parent summary as reference
ref_path = f'{DATA_DIR}/tag_parent_counts.csv'
parent_summary.reset_index().to_csv(ref_path, index=False)
print(f'Saved parent summary → {ref_path}')

Saved 1,495 pairs → ../data/tag_parents.csv
Saved parent summary → ../data/tag_parent_counts.csv


### 5. Coverage stats

How many albums gain a parent-level signal from this mapping?

In [7]:
# Albums covered by at least one child tag that has a parent
child_tag_ids = (
    tags[tags['name'].isin(parent_map['child'])]['id'].values
)
albums_with_parent = album_tags[album_tags['tag_id'].isin(child_tag_ids)]['album_id'].nunique()
total_tagged = album_tags['album_id'].nunique()

print(f'Albums with at least one child tag : {albums_with_parent:,}')
print(f'Total tagged albums                : {total_tagged:,}')
print(f'Coverage                           : {albums_with_parent/total_tagged*100:.1f}%')

# Distribution of parent genres by album count
parent_name_to_id = tags.set_index('name')['id'].to_dict()
top_parents = parent_summary.head(15).reset_index()
top_parents['parent_id'] = top_parents['parent'].map(parent_name_to_id)
print('\n=== Top 15 parent genres by child album coverage ===')
print(top_parents[['parent', 'n_children', 'total_child_albums']].to_string(index=False))

Albums with at least one child tag : 519,640
Total tagged albums                : 1,011,339
Coverage                           : 51.4%

=== Top 15 parent genres by child album coverage ===
      parent  n_children  total_child_albums
        rock          63              163875
       metal          42               80438
       indie           6               51553
 alternative          24               45744
       synth           7               32243
 progressive          17               24237
       house          19               24151
        jazz          27               22358
contemporary          14               21712
         pop          38               21173
 psychedelic           8               21118
       blues          27               18999
     ambient          16               18023
        punk          23               16276
        wave           8               14157


In [8]:
import pandas as pd

DATA_DIR = '../data'
parent_map = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')

# Inspect any parent's children
def show_children(parent_name, n=30):
    children = parent_map[parent_map['parent'] == parent_name].sort_values('child_albums', ascending=False)
    print(f'\n=== "{parent_name}" ({len(children)} children) ===')
    print(children[['child', 'child_albums']].head(n).to_string(index=False))

for p in ['wave', 'contemporary', 'synth', 'indie', 'progressive', 'psychedelic', 'ambient']:
    show_children(p)


=== "wave" (8 children) ===
         child  child_albums
      new wave         13261
       no wave           802
     post-wave            24
third wave ska            16
     cold wave            15
        x-wave            14
permanent wave            13
   trap / wave            12

=== "contemporary" (14 children) ===
                 child  child_albums
     contemporary jazz         12409
      contemporary r&b          4117
contemporary christian          1135
    adult contemporary          1063
     contemporary folk           877
contemporary classical           874
  contemporary country           841
   contemporary gospel           193
      contemporary r b            76
   contemporary reggae            36
      r b contemporary            32
    contemporary blues            24
contemporary bluegrass            23
      contemporary pop            12

=== "synth" (7 children) ===
        child  child_albums
    synth-pop         29497
dungeon synth          2154
   

In [9]:
import pandas as pd

DATA_DIR = '../data'
df = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')

# 1. Delete bad parents entirely
bad_parents = {'wave', 'dark', 'classic', 'music', 'stoner', 'band',
               'songwriter', 'bass', 'improvisation', 'charts', 'space',
               'synth', 'contemporary', 'garage'}
df = df[~df['parent'].isin(bad_parents)]

# 2. Reassign alternative rock and friends to rock
df.loc[df['child'] == 'alternative rock', 'parent'] = 'rock'

# 3. Reassign progressive sub-genres
prog_to_trance = ['progressive trance', 'progressive psytrance']
prog_to_house  = ['progressive house']
prog_to_country = ['progressive country']
prog_to_delete  = ['progressive breaks', 'progressive soul', 'progressive rap',
                   'progressive-rock', 'progressive-metal']  # hyphen duplicates

df.loc[df['child'].isin(prog_to_trance),  'parent'] = 'trance'
df.loc[df['child'].isin(prog_to_house),   'parent'] = 'house'
df.loc[df['child'].isin(prog_to_country), 'parent'] = 'country'
df = df[~df['child'].isin(prog_to_delete)]

# 4. Reassign psychedelic trance
df.loc[df['child'] == 'psychedelic trance', 'parent'] = 'trance'
df = df[df['child'] != 'psychedelic/garage']

# 5. Remove other malformed/niche tags
df = df[df['child'] != 'rock and indie']
df = df[df['child'] != 'offizielle charts']

# 6. Save cleaned map
df.to_csv(f'{DATA_DIR}/tag_parents.csv', index=False)
print(f'Cleaned pairs remaining: {len(df):,}')

# Verify remaining parents
print('\nRemaining parents:')
print(df.groupby('parent')['child_albums'].sum().sort_values(ascending=False).head(20))

Cleaned pairs remaining: 1,366

Remaining parents:
parent
rock            202276
metal            80438
indie            51006
house            28613
jazz             22358
pop              21173
psychedelic      21001
blues            18999
ambient          18023
punk             16276
progressive      14858
hardcore         12332
country          11420
trance           10815
classical        10521
alternative       7343
dance             6266
experimental      6223
noise             5657
instrumental      5565
Name: child_albums, dtype: int64
